Transform Results Data

1. Read bronze_results table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
4. Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)
5. Filter out rows where season, round, constructor_id, or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of column race_name to Title Case
8. Write the transformed data to silver_results table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
bronze_table = F"{catalog_name}.{bronze_schema}.results"
silver_table = F"{catalog_name}.{silver_schema}.results"

In [0]:
results_df = spark.read.table(bronze_table)


In [0]:
display(results_df)

In [0]:
from pyspark.sql import functions as F

In [0]:
results_selected_df = (
    results_df.select(
        F.col("date"),
        F.col("raceName"),
        F.col("round"),
        F.col("season"),
        F.col("url"),
        F.col("constructorId"),
        F.col("driverId"),
        F.col("grid"),
        F.col("laps"),
        F.col("number"),
        F.col("points"),
        F.col("position"),
        F.col("positionText"),
        F.col("status"),
        F.col("timestamp"),
        F.col("source_file")    
    )
)



In [0]:
display(results_selected_df)

### Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
### Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)

In [0]:
results_renamed_df = (
    results_selected_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "positionText": "finish_position_text",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "finish_position",
        })

)



In [0]:
display(results_renamed_df)

### Filter out rows where season, round, constructor_id, or driver_id is null (business key validation)

In [0]:
results_valid_df =(
    results_renamed_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )    
)

In [0]:
display(results_renamed_df.count() - results_valid_df.count())


### Remove duplicate records

In [0]:
results_distinct_df = results_renamed_df.dropDuplicates(["season", "round", "driver_id", "constructor_id"])


In [0]:
display(results_distinct_df)

In [0]:
display(results_valid_df.count() - results_distinct_df.count())

### Transform values of column race_name to Title Case

In [0]:
results_final_df = (
    results_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)        

In [0]:
display(results_final_df)

In [0]:
(
    results_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)